In [1]:
import torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video
import yaml

/home/ey561504/projects/hackathon/Global-MIT-AI-Hackathon/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Available models: Wan-AI/Wan2.1-T2V-14B-Diffusers, Wan-AI/Wan2.1-T2V-1.3B-Diffusers
model_id = "Wan-AI/Wan2.1-T2V-14B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16)
pipe.to("cuda")

Loading pipeline components...: 100%|██████████| 5/5 [00:59<00:00, 11.92s/it]


WanPipeline {
  "_class_name": "WanPipeline",
  "_diffusers_version": "0.33.1",
  "_name_or_path": "Wan-AI/Wan2.1-T2V-14B-Diffusers",
  "scheduler": [
    "diffusers",
    "UniPCMultistepScheduler"
  ],
  "text_encoder": [
    "transformers",
    "UMT5EncoderModel"
  ],
  "tokenizer": [
    "transformers",
    "T5TokenizerFast"
  ],
  "transformer": [
    "diffusers",
    "WanTransformer3DModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKLWan"
  ]
}

In [3]:
with open("../prompts.yaml", "r") as file:
    prompts = yaml.safe_load(file)

prompt = prompts["brainrot_1"]

negative_prompt = prompts["negative_2"]

In [4]:
output = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=832,
    width=480,
    num_frames=81,
    guidance_scale=3.0,
)

100%|██████████| 50/50 [08:36<00:00, 10.34s/it]


In [5]:
videos = output.frames
export_to_video(videos[0], "../output/brainrot1.mp4", fps=16)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


'../output/brainrot1.mp4'

### Cuda Stuff

In [6]:
print(f"Allocated Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved Memory: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
print(f"Max Allocated Memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
print(f"Max Reserved Memory: {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")

Allocated Memory: 40.67 GB
Reserved Memory: 54.11 GB
Max Allocated Memory: 50.10 GB
Max Reserved Memory: 54.11 GB


In [7]:
torch.cuda.empty_cache()